### Applications diverses : manipulation du modèle multi-sites et multi-réplicats

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
folder_name = 'Tunis_F2/Day_3/src/'
root_dir = '/content/gdrive/My Drive/'
base_dir = root_dir + folder_name

In [ ]:
pip install scienceplots

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import os 
import sys

current_dir = os.getcwd()
module_path = os.path.abspath(os.path.join(base_dir, '..', 'libs'))
if module_path not in sys.path:
    sys.path.append(module_path)

from SCOU_extended_3_1 import *
from utils import *

In [ ]:
data = pd.read_csv(base_dir.split('src')[0] + '/data/ammonium_dataset.csv', sep=";")
data.dateStart = pd.to_datetime(data.dateStart)

In [ ]:
data

- Q1 : Traitez le signal obs des 4 stations du jeu de données en prenant en compte les effets de dilution grâce au volume. Utilisez les fonctions model_fit et get_results. La lod correspondante à la colonne 'obs' est la colonne 'lod'. Pour le moment, n'utilisez pas les colonnes 'obs_2' et 'lod_2'. Complétez les deux cellules suivantes :

In [ ]:
def data_processing(data, wwtp):
    sub_data = data.loc[data.plantName==wwtp].copy()
    sub_data = sub_data.drop('plantName', axis=1)

    # convertissez la colonne dateStart en objet datetime:
    # convertissez les colonnes plantVolume et meanVolume en L (elles sont originellement exprimées en m3):
    # pour toute valeur de obs inférieure à la lod, remplacer cette valeur par la valeur de la lod au même index. idem pour obs_2
    # et lod_2:
    # utilisez plantVolume et meanVolume pour corriger les effets de dilution sur les colonnes obs et lod (et _2). 
    # hint : on continuera à travailler en concentration, pas en flux.
    # convertissez ces colonnes en échelle logarithmique, base décimale:
    # rajoutez les jours manquants au calendrier:
    return sub_data

In [ ]:
TUNING_ITERS = 1000
SAMPLING_ITERS = 1000
NB_CHAINS = 5

In [ ]:
wwtp_results = {}

for idx, wwtp in enumerate(data.plantName.unique()):
    print(wwtp)
    sub_data = data_processing(data, wwtp)
    # utilisez les fonctions model_fit et get_results pour ajouter les résultats du lissage au dataframe sub_data

    wwtp_results[wwtp] = sub_data.copy()

- Q2 : Même question, en utilisant l'information de l'ammonium (NH4)

- Q3 : Utilisez le lisseur pour proposer un signal qui agrège l'information à l'échelle de toute la métropole de Paris (utilisez le signal traité avec le volume). Attention, il ne s'agit pas ici d'utiliser l'approche que vous aviez suggérée lors du TP1 ! On utilisera la fonction model_fit en lui donnant comme observations les colonnes 'obs' de chaque station (donc 4 colonnes) et les colonnes 'lod' de chaque station (donc 4 colonnes également, dans le même ordre que les colonnes 'obs').

- Q4 : Comparez l'estimation obtenue à celle que vous aviez proposée lors du TP1

In [ ]:
for idx, wwtp in enumerate(data.plantName.unique()):
    print(wwtp)
    sub_data = data_processing(data, wwtp)
    # créez un dataframe concaténé qui contient les colonnes suivantes :
    # datestart | obs_PARIS_MARNE_AVAL | obs_PARIS_SEINE-AMONT | obs_PARIS_SEINE-MOREE | obs_PARIS_SEINE-CENTRE | 
    #           | lod_PARIS_MARNE_AVAL | lod_PARIS_SEINE-AMONT | lod_PARIS_SEINE-MOREE | lod_PARIS_SEINE-CENTRE | 
    # on les nommera avec l'indice 
    # 0 pour PARIS_MARNE_AVAL
    # 1 pour PARIS_SEINE-AMONT
    # 2 pour PARIS_SEINE-MOREE
    # 3 pour PARIS_SEINE-CENTRE

In [ ]:
TUNING_ITERS = 1000
SAMPLING_ITERS = 1000
NB_CHAINS = 5

In [ ]:
#obs_columns = ...
#lod_columns = ...

scou = model_fit(full_df, obs_columns, lod_columns,
                 TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS,
                 NB_CHAINS=NB_CHAINS, n_sites=4, n_replicates=1
                )

In [ ]:
get_results(full_df, scou, remove_those = [0, 3, 4], NB_CHAINS=NB_CHAINS)

In [ ]:
wwtps = data.plantName.unique().tolist()

In [ ]:
with plt.style.context(['science', 'notebook', 'grid']):

    LABEL_SIZE = 30
    TICK_SIZE = 30
    TITLE_SIZE = 38
    LEGEND_SIZE = 30
    DATES_SIZE = 18
    figsize = (28, 14) #figsize = (32, 10)
    
    plt.rc('axes', labelsize=LABEL_SIZE)
    plt.rc('xtick', labelsize=TICK_SIZE)   
    plt.rc('ytick', labelsize=TICK_SIZE)
    plt.rc('figure', titlesize=TITLE_SIZE)
    plt.rc('legend', fontsize=LEGEND_SIZE)
    plt.rcParams['text.usetex'] = True
    
    fig = plt.figure(figsize=figsize, layout="constrained")
    
    ax_dict = fig.subplot_mosaic(
        """
        A
        """
    )
    nsites = 4
    colors = ['dodgerblue', 'red', 'darkorange', 'forestgreen', 'blueviolet', 'mediumorchid', 'navy', 'teal',
         'gold', 'lightseagreen', 'limegreen', 'fuchsia', 'mediumturquoise', 'silver', 'coral', 'olive', 'black']

    for letter in ['A']:
        sub_data = full_df.copy()
        ### A    
        for i in range(nsites):
            scatter_points = ax_dict[letter].scatter(sub_data.dateStart.values, sub_data[f'obs_{i}'].values, label=f'{wwtps[i]}', 
                                 color=colors[i], edgecolor='black', s=360, zorder=3,
                                 linewidths=1.5, alpha=0.9, vmin=0, vmax=1)
    
            lod_points = ax_dict[letter].scatter(sub_data.loc[sub_data[f'obs_{i}']<=sub_data[f'lod_{i}']].dateStart.values,
                                              sub_data.loc[sub_data[f'obs_{i}']<=sub_data[f'lod_{i}'], f'obs_{i}'].values,
                                 color='none', edgecolor='red', s=520, zorder=2, linewidth=5)

     
        ax_dict[letter].plot(sub_data.dateStart.values, sub_data.muX.values, label='Grand Paris', color='limegreen', linewidth=10, zorder=3)
        ax_dict[letter].plot(sub_data.dateStart.values, sub_data.muX.values, color='black', linewidth=3, zorder=3)               
        ax_dict[letter].fill_between(sub_data.dateStart.values, sub_data.ICL.values, sub_data.ICU.values, alpha=.3, color='green')

        
        ax_dict[letter].set_ylabel("Concentration (GU/L) - $\log_{10}$ scale")
        ax_dict[letter].set_xlabel("Sampling date")
        ax_dict[letter].tick_params(axis='x', labelsize=TICK_SIZE, rotation=45)
        ax_dict[letter].tick_params(axis='y', labelsize=TICK_SIZE)
        ax_dict[letter].grid(linewidth=1, color='black', alpha=0.8)
        ax_dict[letter].set_title('Grand Paris', size=TITLE_SIZE)


    # Main legend
    plt.rcParams['text.usetex'] = False
    h1, l1 = ax_dict[letter].get_legend_handles_labels()
    fig.legend(h1, l1, loc='upper center', bbox_to_anchor=(0.5, 0), fancybox=True, shadow=True, ncol=5)
    #plt.savefig("../outputs/figs/2026-06-01_SARS_Msites_12.pdf", bbox_inches = 'tight')
    
    plt.show()

- Q5 : Utilisez maintenant les deux séries d'observations et de lod pour chaque station, et reprenez les questions 3 et 4 !